# Session 38: Gradient Boosting

Data Science with Python, Level 3, Stage 6: Applied ML

## Learning Objectives

By the end of this session you will be able to:

1. Explain how boosting differs from a forest.
2. Watch a boosted model improve tree by tree.
3. Tune learning rate, depth and number of trees.
4. Use early stopping to choose the number of trees.
5. Use native categorical support.
6. Know where XGBoost and LightGBM fit in.

## Where We Are

A random forest grows its trees **independently** and averages them. **Gradient boosting** grows trees **one after another**: each new small tree is trained on the mistakes the previous trees still make. Step by step the errors shrink.

Boosting libraries such as XGBoost and LightGBM win many data science competitions on tabular data. scikit-learn has a fast version built in, `HistGradientBoosting`, which we use today. Boosting is powerful, but it overfits if you are careless, so tuning matters.

## Setup

Run this cell first. It loads the libraries and sets the address of the course datasets.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import (GradientBoostingRegressor, HistGradientBoostingClassifier,
                              HistGradientBoostingRegressor, RandomForestClassifier)
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import roc_auc_score, r2_score, mean_absolute_error

DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
bikes = pd.read_csv(DATA_URL + "bike_rentals.csv")
churn = pd.read_csv(DATA_URL + "telecom_churn.csv")
churn["left"] = (churn["churn"] == "Yes").astype(int)
churn["monthly_contract"] = (churn["contract"] == "Month-to-month").astype(int)
churn["fiber"] = (churn["internet"] == "Fiber").astype(int)
churn["echeck"] = (churn["payment_method"] == "Electronic check").astype(int)
features = ["tenure_months", "monthly_charges", "support_calls", "senior",
            "monthly_contract", "fiber", "echeck"]
X_train, X_test, y_train, y_test = train_test_split(
    churn[features].astype(float), churn["left"],
    test_size=0.25, random_state=42, stratify=churn["left"])

## 1. How Boosting Works

In [ ]:
cols = ["temp_c", "humidity", "wind_kmh", "working_day", "holiday"]
Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    bikes[cols].astype(float), bikes["rentals"], test_size=0.25, random_state=0)

boost = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                  max_depth=3, random_state=0)
boost.fit(Xb_train, yb_train)

In [ ]:
for i, pred in enumerate(boost.staged_predict(Xb_test), start=1):
    if i in [1, 10, 50, 100, 200, 300]:
        print(f"{i:3} trees: R2 {r2_score(yb_test, pred):.3f}"
              f"  MAE {mean_absolute_error(yb_test, pred):5.0f}")

## 2. Boosting Can Overfit

In [ ]:
for lr, depth in [(0.1, None), (0.3, None), (0.05, 3)]:
    m = HistGradientBoostingClassifier(learning_rate=lr, max_depth=depth,
                                       max_iter=200, early_stopping=False,
                                       random_state=0).fit(X_train, y_train)
    auc = roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])
    print(lr, depth, "train acc", round(m.score(X_train, y_train), 3), "AUC", round(auc, 3))

| Setting | What it controls | Typical values |
| --- | --- | --- |
| learning_rate | How much each tree corrects | 0.01 to 0.2 |
| max_iter / n_estimators | Number of trees | 100 to 2000, with early stopping |
| max_depth / max_leaf_nodes | Size of each tree | Depth 3 to 6 |
| min_samples_leaf | Smallest group a leaf may hold | 20 or more |
| l2_regularization | Penalty on large corrections | 0 to 1 |

Lower learning rate needs more trees. Early stopping finds the right number automatically.

In [ ]:
model = HistGradientBoostingClassifier(learning_rate=0.05, max_iter=1000,
                                       early_stopping=True, validation_fraction=0.2,
                                       n_iter_no_change=20, random_state=0)
model.fit(X_train, y_train)
print("Trees used:", model.n_iter_)
print("Test AUC:", round(roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]), 3))

## 3. Categories Without Encoding

In [ ]:
Xc = bikes[["season", "weather", "temp_c", "humidity", "wind_kmh",
            "working_day", "holiday"]].astype({"season": "category",
                                              "weather": "category"})
model = HistGradientBoostingRegressor(categorical_features="from_dtype")
folds = KFold(n_splits=5, shuffle=True, random_state=0)
print("R2:", cross_val_score(model, Xc, bikes["rentals"], cv=folds).mean().round(3))

## 4. XGBoost and LightGBM

In [ ]:
# XGBoost is preinstalled in Colab. Elsewhere: pip install xgboost
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3,
                    subsample=0.8, colsample_bytree=0.8)
xgb.fit(X_train, y_train)
print("AUC:", roc_auc_score(y_test, xgb.predict_proba(X_test)[:, 1]))

| Tool | Strength | When to use |
| --- | --- | --- |
| HistGradientBoosting | Built into scikit-learn, fast, handles missing values | Default choice in this course |
| XGBoost | Very popular, many options, GPU support | Competitions, large data |
| LightGBM | Very fast on big tables | Millions of rows |
| CatBoost | Strong with many category columns | Lots of text categories |

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Fast learning | A high learning rate with many trees | Use 0.05 to 0.1 and early stopping. |
| Deep trees | Boosting with deep trees | Keep trees shallow: depth 3 to 6. |
| Tuning on test | Picking the number of trees from test scores | Use early stopping or cross-validation. |
| Skipping baselines | Assuming boosting always wins | Compare with logistic regression and a forest. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: Learning Curve Chart

Using `boost.staged_predict`, compute the test MAE after every tree (1 to 300) for the bike model and plot it as a line.

In [ ]:
# Write your code below

### In-Class Exercise 2: Learning Rate Race

Fit `GradientBoostingRegressor` on the bike data with learning rates 0.01, 0.1 and 0.5 (300 trees, depth 3). Print the test R2 for each.

In [ ]:
# Write your code below

### In-Class Exercise 3: Three Models, One Table

On the churn split, compare the test AUC of a random forest (300 trees, `min_samples_leaf=10`), the default `HistGradientBoostingClassifier`, and the shallow slow version (`learning_rate=0.05`, `max_depth=3`, `max_iter=150`).

In [ ]:
# Write your code below

### In-Class Exercise 4: Missing Values Welcome

`HistGradientBoosting` accepts missing values. Fit it on the churn columns `tenure_months`, `monthly_charges`, `total_charges` and `support_calls` without imputing (11 values are missing) and report the cross-validated AUC.

In [ ]:
# Write your code below

## Summary

- Boosting adds shallow trees in sequence, each fixing remaining errors.
- A forest averages independent trees; boosting corrects step by step.
- Lower learning rates and shallow trees reduce overfitting.
- Early stopping chooses the number of trees for you.
- HistGradientBoosting handles categories and missing values natively.
- XGBoost and LightGBM follow the same fit and predict pattern.

## What's Next

So far every model had a target to predict. Next session there is no target at all: you will group mall customers into segments with K-Means, choose the number of groups, and describe each segment for the marketing team.